# CV Research Coach

This notebook implements a CV Deep Research Agent using LangGraph.

Key design choices:
- Parallel fan-out after CV extraction: ATS review and research planning run on independent branches
- Conditional company branch: company research is only executed when a company is provided
- Synthesize join node: merges all parallel branches before gap analysis
- Research confidence loop: gap analysis can route back to deepen_research if evidence is thin
- 3-way evaluator router: evaluator can send report back to deeper research OR to report drafting, not just one

In [ ]:
# Imports

import json
import os
import re
from typing import Annotated, Any, Dict, List, Optional

import asyncio
import nest_asyncio
nest_asyncio.apply()

_original_run = asyncio.run
def _patched_run(main, kwargs):
    kwargs.pop("loop_factory", None)
    return _original_run(main, kwargs)
asyncio.run = _patched_run

from dotenv import load_dotenv
from pypdf import PdfReader
from pydantic import BaseModel, Field
from IPython.display import display, Markdown, Image

from langchain.agents import Tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

from typing_extensions import TypedDict

In [ ]:
load_dotenv(override=True)

## 1 Structured Output Schemas

These Pydantic models define the typed data that flows through the graph's state.

In [ ]:
# Research plan

class ResearchItem(BaseModel):
    reason: str = Field(description="Why this search is important for the CV review.")
    query: str = Field(description="The search term to use for the web search.")


class ResearchPlan(BaseModel):
    role_searches: list[ResearchItem] = Field(description="Searches about the target role: skills, keywords, trends.")
    company_searches: list[ResearchItem] = Field(description="Searches about the target company (empty list if no company provided).")


# Gap analysis

class GapItem(BaseModel):
    area: str = Field(description="The gap area: skills, keywords, experience, metrics, or structure.")
    severity: str = Field(description="critical, major, or minor.")
    detail: str = Field(description="What is missing or weak.")
    fix: str = Field(description="Specific suggestion to close this gap.")


class GapAnalysis(BaseModel):
    overall_fit_score: int = Field(description="How well the CV fits the target role, 0-100.")
    gaps: list[GapItem] = Field(description="All gaps found between the CV and the target role/market.")
    strengths: list[str] = Field(description="What the CV already does well for this role.")


# Rewrite suggestion

class RewriteSuggestion(BaseModel):
    original: str = Field(description="The original bullet point or section from the CV.")
    improved: str = Field(description="The improved version with stronger impact and keywords.")
    reason: str = Field(description="Why this rewrite is better.")


# Final report

class FinalReport(BaseModel):
    executive_summary: str = Field(description="2-3 sentence overview of the CV review findings.")
    ats_readiness_score: int = Field(description="ATS compliance score out of 100.")
    role_fit_score: int = Field(description="How well the CV matches the target role, 0-100.")
    market_insights: str = Field(description="Key findings from market research about the target role.")
    company_insights: str = Field(description="Key findings about the target company (or 'N/A' if none).")
    gap_summary: str = Field(description="Summary of the most critical gaps found.")
    rewrite_suggestions: list[RewriteSuggestion] = Field(description="At least 3 specific bullet point rewrites.")
    keywords_to_add: list[str] = Field(description="Keywords the candidate should weave into the CV.")
    interview_prep_topics: list[str] = Field(description="5+ topics the candidate should prepare for interviews.")
    thirty_day_action_plan: list[str] = Field(description="Ordered steps the candidate should take in the next 30 days.")


# Evaluator output

class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Specific feedback on what the report is missing or doing wrong.")
    report_is_acceptable: bool = Field(description="True only if the report has enough evidence, specific rewrites, and actionable gaps.")
    needs_more_research: bool = Field(description="True if the report's weakness is due to insufficient market/company research (not just bad writing). False if the report just needs a rewrite.")


# Research confidence output

class ResearchConfidence(BaseModel):
    confidence: str = Field(description="'high' if the research gives enough evidence for solid gap analysis, 'low' if research is too thin or vague.")
    missing_areas: list[str] = Field(description="Specific topics the research failed to cover well enough.")

print("Schemas defined ")

## 2 Tools

In [ ]:
# CV extraction

def extract_cv_text(file_path: str) -> str:
    """Read a CV/resume from a PDF or text file and return its text content as JSON."""
    try:
        if file_path.strip().endswith(".pdf"):
            reader = PdfReader(file_path.strip())
            text = ""
            for page in reader.pages:
                t = page.extract_text()
                if t:
                    text += t
        else:
            with open(file_path.strip(), "r", encoding="utf-8") as f:
                text = f.read()
        return json.dumps({"success": True, "text": text, "word_count": len(text.split())})
    except FileNotFoundError:
        return json.dumps({"success": False, "error": f"File not found: {file_path}"})


# ATS formatting check

def check_ats_formatting(cv_text: str) -> str:
    """Run ATS compliance checks on CV text and return results as JSON."""
    results = {}
    standard_sections = ["summary", "experience", "education", "skills", "contact"]
    results["found_sections"] = [s for s in standard_sections if s in cv_text.lower()]
    results["missing_sections"] = [s for s in standard_sections if s not in cv_text.lower()]

    word_count = len(cv_text.split())
    results["word_count"] = word_count
    results["length_verdict"] = "too short" if word_count < 300 else "too long" if word_count > 1000 else "good"

    results["has_email"] = bool(re.search(r'\b[\w.-]+@[\w.-]+\.\w+\b', cv_text))
    results["has_phone"] = bool(re.search(r'[\+\(]?[0-9][\d \-\(\)]{7,}\d', cv_text))

    action_verbs = ["led", "managed", "developed", "designed", "implemented", "achieved",
                    "increased", "reduced", "created", "launched", "built", "delivered",
                    "optimized", "streamlined", "mentored", "spearheaded", "orchestrated"]
    found_verbs = [v for v in action_verbs if v in cv_text.lower()]
    results["action_verbs_found"] = found_verbs
    results["action_verb_count"] = len(found_verbs)

    results["has_quantified_achievements"] = bool(re.search(r'\d+%|\$[\d,]+|\d+\+', cv_text))
    return json.dumps(results)


# Keyword match

def check_keyword_match(cv_text: str, job_keywords: str) -> str:
    """Compare CV text against comma-separated job requirement keywords. Returns match results as JSON."""
    cv_lower = cv_text.lower()
    keywords = [kw.strip().lower() for kw in job_keywords.split(",") if kw.strip()]
    matched = [kw for kw in keywords if kw in cv_lower]
    missing = [kw for kw in keywords if kw not in cv_lower]
    score = round(len(matched) / len(keywords) * 100, 1) if keywords else 0
    return json.dumps({"matched_keywords": matched, "missing_keywords": missing, "match_score_pct": score})


# Web search (Serper)

serper = GoogleSerperAPIWrapper()


# Wrap everything as LangChain Tool objects for ToolNode
tool_extract_cv = Tool(name="extract_cv_text", func=extract_cv_text,
    description="Read a CV/resume from a PDF or text file. Input: file_path string.")

tool_ats_check = Tool(name="check_ats_formatting", func=check_ats_formatting,
    description="Run ATS compliance checks on CV text. Input: the full CV text string.")

tool_keyword_match = Tool(name="check_keyword_match", func=check_keyword_match,
    description="Compare CV text against job keywords. Input: 'cv_text|||job_keywords' (pipe-separated).")

tool_search = Tool(name="web_search", func=serper.run,
    description="Search the web for current information. Input: a search query string.")

all_tools = [tool_extract_cv, tool_ats_check, tool_keyword_match, tool_search]

print(f"Tools defined: {[t.name for t in all_tools]}")

## 3 LangGraph State

In [ ]:
class CVResearchState(TypedDict):
    """Shared state for the CV research graph."""

    # Chat history
    messages: Annotated[list, add_messages]

    # Inputs
    cv_file: str
    target_role: str
    company: str               
    job_keywords: str           # comma-separated or empty

    # Stage outputs
    cv_text: str
    ats_results: str            # JSON string from tool
    keyword_results: str        # JSON string from tool
    research_plan: Optional[str]  
    role_research: str
    company_research: str
    gap_analysis: Optional[str]   

    # Research deepening
    research_confidence: str    # 'high' or 'low'
    research_deepen_count: int  # how many times we've looped back for more research

    # Report
    draft_report: Optional[str]   # JSON serialized FinalReport
    evaluator_feedback: Optional[str]
    evaluator_needs_research: bool  # True if evaluator says weakness is research, not writing
    report_is_acceptable: bool
    revision_count: int

print("State schema defined ")

## 4 LLM Setup

We use two `ChatOpenAI` instances:
- A worker LLM with tools bound (for nodes that need to call tools or generate content)
- An evaluator LLM with structured output for judging report quality

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
llm_with_tools = llm.bind_tools(all_tools)

evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

# Higher-quality LLM for the report-writing step
report_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)

print("LLMs configured ")

## 5 Graph Nodes

In [ ]:
# Node 1: Validate Input

def validate_input(state: CVResearchState) -> Dict[str, Any]:
    """Check that cv_file and target_role are present. Raise early if not."""
    cv_file = state.get("cv_file", "").strip()
    target_role = state.get("target_role", "").strip()
    errors = []
    if not cv_file:
        errors.append("Missing cv_fileprovide a path to a PDF or TXT resume.")
    if not target_role:
        errors.append("Missing target_rolespecify the job you're targeting.")
    if errors:
        error_msg = "Input validation failed: " + "; ".join(errors)
        raise ValueError(error_msg)
    print(f"  Input validCV: {cv_file}, Role: {target_role}")
    return {"messages": [HumanMessage(content=f"Review CV at '{cv_file}' for the role of {target_role}.")]}

In [ ]:
# Node 2: Extract CV Text

def extract_cv(state: CVResearchState) -> Dict[str, Any]:
    """Read the CV file and put the full text into state."""
    result_json = extract_cv_text(state["cv_file"])
    result = json.loads(result_json)
    if not result.get("success"):
        raise FileNotFoundError(result.get("error", "Could not read CV file."))
    print(f"  CV extracted{result['word_count']} words")
    return {"cv_text": result["text"]}

In [ ]:
# Node 3: ATS Review

def ats_review(state: CVResearchState) -> Dict[str, Any]:
    """Run ATS formatting checks and optional keyword match."""
    ats = check_ats_formatting(state["cv_text"])
    kw_result = ""
    if state.get("job_keywords", "").strip():
        kw_result = check_keyword_match(state["cv_text"], state["job_keywords"])
    print(f"  ATS review complete")
    return {"ats_results": ats, "keyword_results": kw_result}

In [ ]:
# Node 4: Plan Research

HOW_MANY_ROLE_SEARCHES = 3
HOW_MANY_COMPANY_SEARCHES = 2

def plan_research(state: CVResearchState) -> Dict[str, Any]:
    """Use the LLM to produce a structured research plan."""
    planner_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2).with_structured_output(ResearchPlan)

    company_instruction = ""
    if state.get("company", "").strip():
        company_instruction = f"The target company is: {state['company']}. Plan {HOW_MANY_COMPANY_SEARCHES} company-specific searches."
    else:
        company_instruction = "No target company was specified. Return an empty list for company_searches."

    messages = [
        SystemMessage(content=f"""You are a career research planner. Given a target job role and optionally a company name,
produce a research plan with:
- {HOW_MANY_ROLE_SEARCHES} web searches about the role (in-demand skills, common job keywords, salary trends, market outlook)
- Company searches as instructed below

Make searches specific and current. Include the year 2025/2026 in queries for freshness."""),
        HumanMessage(content=f"Target role: {state['target_role']}\n{company_instruction}")
    ]
    plan: ResearchPlan = planner_llm.invoke(messages)
    total = len(plan.role_searches) + len(plan.company_searches)
    print(f"  Research planned{total} searches ({len(plan.role_searches)} role + {len(plan.company_searches)} company)")
    return {"research_plan": plan.model_dump_json()}

In [ ]:
# Node 5: Run Role Research

def run_role_research(state: CVResearchState) -> Dict[str, Any]:
    """Execute all role-related web searches from the research plan."""
    plan = ResearchPlan.model_validate_json(state["research_plan"])
    results = []
    for item in plan.role_searches:
        try:
            raw = serper.run(item.query)
            # Ask LLM to summarize the search result concisely
            summary = llm.invoke([
                SystemMessage(content="Summarize these web search results in 2-3 concise paragraphs (under 200 words). Focus on facts relevant to a job seeker."),
                HumanMessage(content=f"Query: {item.query}\nReason: {item.reason}\n\nRaw results:\n{raw}")
            ])
            results.append(f"{item.query}\n{summary.content}")
        except Exception as e:
            results.append(f"{item.query}\nSearch failed: {e}")
    print(f"  Role research complete{len(results)} searches")
    return {"role_research": "\n\n---\n\n".join(results)}

In [ ]:
# Node 6: Run Company Research (conditional only when company provided)

def run_company_research(state: CVResearchState) -> Dict[str, Any]:
    """Execute company-related web searches. Only called when a company is given."""
    plan = ResearchPlan.model_validate_json(state["research_plan"])
    results = []
    for item in plan.company_searches:
        try:
            raw = serper.run(item.query)
            summary = llm.invoke([
                SystemMessage(content="Summarize these web search results in 2-3 concise paragraphs (under 200 words). Focus on facts relevant to a job seeker targeting this company."),
                HumanMessage(content=f"Query: {item.query}\nReason: {item.reason}\n\nRaw results:\n{raw}")
            ])
            results.append(f"{item.query}\n{summary.content}")
        except Exception as e:
            results.append(f"{item.query}\nSearch failed: {e}")
    print(f"  Company research complete{len(results)} searches")
    return {"company_research": "\n\n---\n\n".join(results)}


def skip_company_research(state: CVResearchState) -> Dict[str, Any]:
    """No-op node when no company is supplied."""
    print(" No company specified, skipping company research")
    return {"company_research": "No target company was specified."}

In [ ]:
# Node 7: Synthesize (join node for parallel branches)

def synthesize(state: CVResearchState) -> Dict[str, Any]:
    """Merge point: all parallel branches have written to state. Log what arrived."""
    has_ats = bool(state.get("ats_results", ""))
    has_role = bool(state.get("role_research", ""))
    has_company = bool(state.get("company_research", ""))
    print(f"  Synthesize ATS: {has_ats}, Role research: {has_role}, Company research: {has_company}")
    return {}  # nothing new to write, just a join point


# Node 8: Analyze Gaps

def analyze_gaps(state: CVResearchState) -> Dict[str, Any]:
    """Compare the CV against market research and produce a structured gap analysis."""
    gap_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2).with_structured_output(GapAnalysis)

    messages = [
        SystemMessage(content="""You are a career gap analyst. You receive a CV, ATS check results, and market research.
Analyze every gap between what the CV shows and what the market demands.
Be thoroughfind at least 5 gaps. Rate each by severity (critical, major, minor).
Also identify the candidate's existing strengths."""),
        HumanMessage(content=f"""## CV Text
{state['cv_text'][:3000]}

## ATS Check Results
{state['ats_results']}

## Keyword Match
{state.get('keyword_results', 'No keyword check was performed.')}

## Market Research on Target Role: {state['target_role']}
{state['role_research']}

## Company Research
{state['company_research']}""")
    ]

    gap: GapAnalysis = gap_llm.invoke(messages)
    print(f"  Gap analysis fit score: {gap.overall_fit_score}/100, {len(gap.gaps)} gaps found")
    return {"gap_analysis": gap.model_dump_json()}


# Node 8b: Assess Research Confidence

MAX_DEEPEN = 1  # allow at most 1 research-deepening pass

def assess_research_confidence(state: CVResearchState) -> Dict[str, Any]:
    """After gap analysis, judge whether the research was deep enough."""
    gap_data = GapAnalysis.model_validate_json(state["gap_analysis"])

    # If we already deepened, skip and proceed
    if state.get("research_deepen_count", 0) >= MAX_DEEPEN:
        print(f" Research deepen cap reached, proceeding")
        return {"research_confidence": "high"}

    confidence_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(ResearchConfidence)

    messages = [
        SystemMessage(content="""You assess whether the market research backing a CV gap analysis is sufficient.
Look at the gap items and the research that was done. If gaps mention areas not covered by the research,
or the research is vague/generic, rate confidence as 'low' and list what's missing.
If the research concretely covers the gap areas, rate 'high'."""),
        HumanMessage(content=f"""## Gap Analysis\n{state['gap_analysis']}\n\n## Role Research\n{state['role_research']}\n\n## Company Research\n{state['company_research']}""")
    ]

    result: ResearchConfidence = confidence_llm.invoke(messages)
    print(f"  Research confidence: {result.confidence} (missing: {result.missing_areas})")
    return {"research_confidence": result.confidence}


# Node 8c: Deepen Research

def deepen_research(state: CVResearchState) -> Dict[str, Any]:
    """Run additional targeted searches based on gaps that lacked evidence."""
    # Use the gap analysis to figure out what to search for
    gap_data = GapAnalysis.model_validate_json(state["gap_analysis"])

    # Build targeted queries from critical/major gaps
    queries = []
    for g in gap_data.gaps:
        if g.severity in ("critical", "major"):
            queries.append(f"{state['target_role']} {g.area} requirements 2025 2026")
    queries = queries[:3]  # cap at 3 extra searches

    new_results = []
    for q in queries:
        try:
            raw = serper.run(q)
            summary = llm.invoke([
                SystemMessage(content="Summarize these search results in 1-2 paragraphs. Focus on concrete facts, numbers, and skill names."),
                HumanMessage(content=f"Query: {q}\n\nResults:\n{raw}")
            ])
            new_results.append(f"{q}\n{summary.content}")
        except Exception as e:
            new_results.append(f"{q}\nSearch failed: {e}")

    extra = "\n\n---\n\n".join(new_results)
    # Append to existing role research
    updated_role_research = state.get("role_research", "") + "\n\n---\n\n## DEEPENED RESEARCH\n" + extra
    deepen_count = state.get("research_deepen_count", 0) + 1
    print(f"  Deepened research{len(new_results)} extra searches (pass {deepen_count})")
    return {"role_research": updated_role_research, "research_deepen_count": deepen_count}

In [ ]:
# Node 9: Draft Report

def draft_report(state: CVResearchState) -> Dict[str, Any]:
    """Synthesize everything into a structured FinalReport."""
    report_llm_structured = report_llm.with_structured_output(FinalReport)

    # Include evaluator feedback if this is a revision pass
    revision_note = ""
    if state.get("evaluator_feedback"):
        revision_note = f"""\n\n## IMPORTANT Evaluator Feedback on Previous Draft
Your previous draft was rejected. Address this feedback:\n{state['evaluator_feedback']}
Make sure your revised report fixes these specific issues."""

    gap_data = GapAnalysis.model_validate_json(state["gap_analysis"])

    messages = [
        SystemMessage(content="""You are a senior career consultant writing a comprehensive CV deep research report.
Produce a thorough, actionable report. Your rewrite suggestions must reference actual bullet points
from the CV and provide improved versions that are specific, quantified, and keyword-rich.
Your interview prep topics should be derived from the research, not generic.
Your 30-day action plan should be concrete and ordered by priority."""),
        HumanMessage(content=f"""## CV Text
{state['cv_text'][:3000]}

## ATS Results
{state['ats_results']}

## Keyword Match
{state.get('keyword_results', 'N/A')}

## Gap Analysis
Gaps: {json.dumps([g.model_dump() for g in gap_data.gaps], indent=2)}
Strengths: {gap_data.strengths}
Fit score: {gap_data.overall_fit_score}/100

## Role Research
{state['role_research']}

## Company Research
{state['company_research']}{revision_note}""")
    ]

    report: FinalReport = report_llm_structured.invoke(messages)
    revision = state.get("revision_count", 0) + 1
    print(f"  Report drafted (revision {revision})")
    return {"draft_report": report.model_dump_json(), "revision_count": revision}

In [ ]:
# Node 10: Evaluate Report

MAX_REVISIONS = 5

def evaluate_report(state: CVResearchState) -> Dict[str, Any]:
    """Judge whether the drafted report is good enough, needs a rewrite, or needs more research."""
    # Safety capalways accept after MAX_REVISIONS to avoid infinite loops
    if state.get("revision_count", 0) >= MAX_REVISIONS:
        print(f"  Max revisions ({MAX_REVISIONS}) reachedaccepting report")
        return {"report_is_acceptable": True, "evaluator_feedback": None, "evaluator_needs_research": False}

    report = FinalReport.model_validate_json(state["draft_report"])

    messages = [
        SystemMessage(content="""You evaluate CV research reports. Check whether the report meets these criteria:
1. At least 3 specific rewrite suggestions that reference actual CV text (not generic)
2. Market insights cite concrete data points (numbers, companies, trends)
3. Gap analysis has at least 3 gaps with severity ratings
4. The 30-day action plan has at least 4 concrete, ordered steps
5. Keywords to add are specific to the target role, not generic

If ANY of these criteria are not met, respond with report_is_acceptable=False.
Also determine: is the weakness because the underlying RESEARCH was too thin/vague (needs_more_research=True),
or because the report WRITING was poor despite having good data (needs_more_research=False)?"""),
        HumanMessage(content=f"""Here is the report to evaluate:\n{report.model_dump_json(indent=2)}""")
    ]

    evaluation: EvaluatorOutput = evaluator_llm_with_output.invoke(messages)

    if evaluation.report_is_acceptable:
        print(f"  Report accepted by evaluator")
    else:
        direction = "research" if evaluation.needs_more_research else "rewrite"
        print(f"  Report rejected → {direction} | {evaluation.feedback[:80]}...")

    return {
        "report_is_acceptable": evaluation.report_is_acceptable,
        "evaluator_feedback": evaluation.feedback if not evaluation.report_is_acceptable else None,
        "evaluator_needs_research": evaluation.needs_more_research if not evaluation.report_is_acceptable else False,
    }

## 6 Routing Functions

In [ ]:
def route_company_research(state: CVResearchState) -> str:
    """Branch: do company research if a company was provided, otherwise skip."""
    if state.get("company", "").strip():
        return "run_company_research"
    return "skip_company_research"


def route_research_confidence(state: CVResearchState) -> str:
    """Branch after gap analysis: deepen research if confidence is low, else proceed."""
    if state.get("research_confidence", "high") == "low":
        return "deepen_research"
    return "draft_report"


def route_after_evaluation(state: CVResearchState) -> str:
    """3-way branch: accept / re-research / rewrite."""
    if state.get("report_is_acceptable", False):
        return "END"
    if state.get("evaluator_needs_research", False) and state.get("research_deepen_count", 0) < MAX_DEEPEN:
        return "deepen_research"
    return "draft_report"

## 7 Build the Graph

In [ ]:
# Build the graph

graph_builder = StateGraph(CVResearchState)
# Add nodes
graph_builder.add_node("validate_input", validate_input)
graph_builder.add_node("extract_cv", extract_cv)
graph_builder.add_node("ats_review", ats_review)
graph_builder.add_node("plan_research", plan_research)
graph_builder.add_node("run_role_research", run_role_research)
graph_builder.add_node("run_company_research", run_company_research)
graph_builder.add_node("skip_company_research", skip_company_research)
graph_builder.add_node("synthesize", synthesize)
graph_builder.add_node("analyze_gaps", analyze_gaps)
graph_builder.add_node("assess_research_confidence", assess_research_confidence)
graph_builder.add_node("deepen_research", deepen_research)
graph_builder.add_node("draft_report", draft_report)
graph_builder.add_node("evaluate_report", evaluate_report)

# Entry
graph_builder.add_edge(START, "validate_input")
graph_builder.add_edge("validate_input", "extract_cv")

# Parallel fan-out: extract_cv -> [ats_review, plan_research]
graph_builder.add_edge("extract_cv", "ats_review")
graph_builder.add_edge("extract_cv", "plan_research")

# Research branch: plan -> role_research -> conditional company
graph_builder.add_edge("plan_research", "run_role_research")
graph_builder.add_conditional_edges(
    "run_role_research",
    route_company_research,
    {"run_company_research": "run_company_research", "skip_company_research": "skip_company_research"}
)

# All parallel branches converge at synthesize
graph_builder.add_edge("ats_review", "synthesize")
graph_builder.add_edge("run_company_research", "synthesize")
graph_builder.add_edge("skip_company_research", "synthesize")

# Synthesize → gap analysis → confidence check
graph_builder.add_edge("synthesize", "analyze_gaps")
graph_builder.add_edge("analyze_gaps", "assess_research_confidence")

# Research confidence loop
graph_builder.add_conditional_edges(
    "assess_research_confidence",
    route_research_confidence,
    {"deepen_research": "deepen_research", "draft_report": "draft_report"}
)
graph_builder.add_edge("deepen_research", "analyze_gaps")  # loop back for re-analysis

# Report → evaluate → 3-way router
graph_builder.add_edge("draft_report", "evaluate_report")
graph_builder.add_conditional_edges(
    "evaluate_report",
    route_after_evaluation,
    {"draft_report": "draft_report", "deepen_research": "deepen_research", "END": END}
)

# Compile with MemorySaver
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("Graph compiled ")

In [ ]:
# Visualize the graph

try:
    img_data = graph.get_graph().draw_mermaid_png()
    display(Image(img_data))
except Exception:
    # Fallback: print the mermaid diagram text
    print(graph.get_graph().draw_mermaid())

## 8 Configure Your Inputs

Set your CV file path, target role, and optionally a company and keywords.

In [ ]:
# YOUR INPUTS

CV_FILE = "Emmanuel_Samuel_CV_AI_DS_ML.pdf"  # Path to your CV (PDF or TXT)
TARGET_ROLE = "Machine Learning Engineer"
COMPANY = "Google"                            # Leave empty string "" to skip company research
JOB_KEYWORDS = "python, machine learning, deep learning, MLOps, AWS, data pipelines, leadership"  # Comma-separated

## 9 Run the Graph

In [ ]:
# Run the CV research graph

config = {"configurable": {"thread_id": "cv-research-1"}}

initial_state = {
    "messages": [],
    "cv_file": CV_FILE,
    "target_role": TARGET_ROLE,
    "company": COMPANY,
    "job_keywords": JOB_KEYWORDS,
    "cv_text": "",
    "ats_results": "",
    "keyword_results": "",
    "research_plan": None,
    "role_research": "",
    "company_research": "",
    "gap_analysis": None,
    "research_confidence": "high",
    "research_deepen_count": 0,
    "draft_report": None,
    "evaluator_feedback": None,
    "evaluator_needs_research": False,
    "report_is_acceptable": False,
    "revision_count": 0,
}

print("Starting CV Research Coach...\n")
result = await graph.ainvoke(initial_state, config=config)
report = FinalReport.model_validate_json(result["draft_report"])
print(report)

## Gradio Interface

A simple web UI to upload a CV and run the LangGraph pipeline interactively.

In [ ]:
import gradio as gr
import uuid


async def gradio_review(cv_file, target_role, company, job_kw):
    """Gradio wrapper for the LangGraph CV research pipeline."""
    if not cv_file or not target_role.strip():
        return "Please provide both a CV file and a target role."

    thread_id = f"gradio-{uuid.uuid4().hex[:8]}"
    cfg = {"configurable": {"thread_id": thread_id}}

    state = {
        "messages": [],
        "cv_file": cv_file.name,
        "target_role": target_role.strip(),
        "company": company.strip() if company else "",
        "job_keywords": job_kw.strip() if job_kw else "",
        "cv_text": "",
        "ats_results": "",
        "keyword_results": "",
        "research_plan": None,
        "role_research": "",
        "company_research": "",
        "gap_analysis": None,
        "research_confidence": "high",
        "research_deepen_count": 0,
        "draft_report": None,
        "evaluator_feedback": None,
        "evaluator_needs_research": False,
        "report_is_acceptable": False,
        "revision_count": 0,
    }

    try:
        final = await graph.ainvoke(state, config=cfg)
    except Exception as e:
        return f"Error during research: {e}"

    rpt = FinalReport.model_validate_json(final["draft_report"])

    lines = []
    lines.append(f"EXECUTIVE SUMMARY\n{rpt.executive_summary}\n")
    lines.append(f"ATS Readiness: {rpt.ats_readiness_score}/100")
    lines.append(f"Role Fit: {rpt.role_fit_score}/100")
    lines.append(f"Revisions done: {final['revision_count']}\n")
    lines.append(f"MARKET INSIGHTS\n{rpt.market_insights}\n")
    lines.append(f"COMPANY INSIGHTS\n{rpt.company_insights}\n")
    lines.append(f"GAP SUMMARY\n{rpt.gap_summary}\n")
    lines.append("REWRITE SUGGESTIONS")
    for i, s in enumerate(rpt.rewrite_suggestions, 1):
        lines.append(f"\n{i}. Original: {s.original}")
        lines.append(f"   Improved: {s.improved}")
        lines.append(f"   Why: {s.reason}")
    lines.append(f"\nKEYWORDS TO ADD\n{', '.join(rpt.keywords_to_add)}\n")
    lines.append("INTERVIEW PREP TOPICS")
    for topic in rpt.interview_prep_topics:
        lines.append(f"  - {topic}")
    lines.append("\n30-DAY ACTION PLAN")
    for i, step in enumerate(rpt.thirty_day_action_plan, 1):
        lines.append(f"  {i}. {step}")

    return "\n".join(lines)


with gr.Blocks(title="CV Research CoachLangGraph") as demo:
    gr.Markdown("# CV Research Coach (LangGraph)\nUpload your CV, enter a target role, and get a research-backed career positioning report with evaluator-driven quality checks.")
    with gr.Row():
        cv_input = gr.File(label="Upload CV (PDF or TXT)", file_types=[".pdf", ".txt"])
        with gr.Column():
            role_input = gr.Textbox(label="Target Role", placeholder="e.g., Machine Learning Engineer")
            company_input = gr.Textbox(label="Target Company (optional)", placeholder="e.g., Google")
            kw_input = gr.Textbox(label="Job Keywords (optional, comma-separated)",
                                  placeholder="python, MLOps, AWS, leadership...")
    output = gr.Textbox(label="Deep Research Report", lines=30)
    btn = gr.Button("Research & Review My CV", variant="primary")
    btn.click(fn=gradio_review, inputs=[cv_input, role_input, company_input, kw_input], outputs=output)

demo.launch()